In [ ]:
# --- Vishalam setup: find the code dataset among the inputs, unpack, install (no secrets) ---
import glob, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
def hours_left(limit=11.2):  # Kaggle stops sessions at 12 h; keep a margin for saving outputs
    return max(0.25, limit - (time.time() - T0) / 3600)
WORK, INPUT, CODE = "/kaggle/working", "/kaggle/input", "/tmp/vishalam"

def inputs_with(marker, hint=""):
    """Input directories that contain `marker` (a relative path) and whose path contains `hint`."""
    hits = glob.glob(f"{INPUT}/**/{marker}", recursive=True)
    dirs = [h[: -len(marker)].rstrip("/") for h in hits]
    return sorted(d for d in set(dirs) if hint in d)

if not os.path.isdir(CODE):
    tars = glob.glob(f"{INPUT}/**/vishalam_code.tar.gz", recursive=True)
    if tars:
        os.makedirs(CODE)
        with tarfile.open(tars[0]) as t:
            t.extractall(CODE, filter="data")
    else:  # Kaggle sometimes unpacks archives itself
        roots = [d for d in inputs_with("vishalam/tokenizer/bpe.py") if os.path.exists(f"{d}/requirements.txt")]
        if not roots:
            raise SystemExit("code dataset not attached - run tools/kaggle_driver.py push-code and re-push this kernel")
        shutil.copytree(roots[0], CODE)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready:", open("VERSION").read().strip() if os.path.exists("VERSION") else "code unpacked")


In [ ]:
# --- Evaluate the agent on CPU: shard 2 of 5 kernels x 4 processes ---
TOKENIZER = inputs_with("tokenizer.json", "vishalam-data-1")[0] + "/tokenizer.json"
m1 = inputs_with("m1/test_seen.jsonl", "vishalam-m1-examples")[0] + "/m1"
MODEL = inputs_with("sft/model.pt", "vishalam-sft-m1-125m-")[0] + "/sft/model.pt"
subprocess.run([sys.executable, "-c", "from vishalam.eval.harness import load_humaneval, load_mbpp; "
                "load_humaneval(); load_mbpp()"], check=True)  # download once, not in every process
n = 5 * 4
shards = [2 * 4 + w for w in range(4)]
procs = [subprocess.Popen(
    f"python -m vishalam.eval.run --weights {MODEL} --tokenizer {TOKENIZER} --data {m1} "
    f"--sets test_seen,test_unseen,handwritten,humaneval,mbpp --sandbox subprocess --device cpu --threads 1 "
    f"--shard {i}/{n} --max-hours {hours_left() - 0.5:.2f} --out {WORK}/shard_{i}.json "
    f"> {WORK}/shard_{i}.log 2>&1", shell=True) for i in shards]
while any(p.poll() is None for p in procs):
    time.sleep(600)
    for i in shards:
        lines = open(f"{WORK}/shard_{i}.log").read().splitlines()
        print(f"shard {i}:", lines[-1] if lines else "(starting)", flush=True)
for i in shards:
    print(open(f"{WORK}/shard_{i}.log").read()[-1500:])
done = ",".join(f"{WORK}/shard_{i}.json" for i in shards if os.path.exists(f"{WORK}/shard_{i}.json"))
get_ipython().system(f"python -m vishalam.eval.run --merge {done} --out {WORK}/results.json")
